In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import Counter

from playball import single_AB, base_status, one_innging, single_game, game_simulation


In [2]:
def batting_order(df):

    # OPS 기준 정렬
    df_ops_sorted = df.sort_values(by='OPS', ascending=False).reset_index(drop=True)
    
    # 4번 타자: OPS 1위
    batter_4 = df_ops_sorted.iloc[0]
    
    # 3번 타자: OPS 2위
    batter_3 = df_ops_sorted.iloc[1]
    
    batter_5 = df_ops_sorted.iloc[2]
    
    # 3,4번 제외한 나머지
    excluded_players = [batter_3['Player'], batter_4['Player'],batter_5['Player']]
    remaining_df = df[~df['Player'].isin(excluded_players)]
    
    # 1,2번 타자: OBP 기준 상위 2명
    df_obp_sorted = remaining_df.sort_values(by='OBP', ascending=False).reset_index(drop=True)
    batter_1 = df_obp_sorted.iloc[0]
    batter_2 = df_obp_sorted.iloc[1]
    
    # 이제 제외된 1~4번 타자 제외한 나머지 5명
    excluded_players += [batter_1['Player'], batter_2['Player']]
    remaining_df_final = df[~df['Player'].isin(excluded_players)]
    
    # 5~9번 타자: OPS 기준 정렬
    rest_sorted = remaining_df_final.sort_values(by='OPS', ascending=False).reset_index(drop=True)
    
    # 최종 타순 결합
    final_batting_order = pd.DataFrame([batter_1, batter_2, batter_3, batter_4,batter_5]).reset_index(drop=True)
    final_batting_order = pd.concat([final_batting_order, rest_sorted], ignore_index=True)
    
    return final_batting_order

In [3]:
# LG

# PA, AB, H, 2B, 3B, HR, SO, BB+HBP
data = [
    ['박해민', 237, 187, 43, 6, 2, 0, 51, 37],  # 33+3
    ['김현수', 233, 201, 59, 9, 0, 5, 29, 31],  # 28+3
    ['오스틴', 236, 200, 63, 10, 0,18, 30, 36], # 31+0
    ['문보경', 259, 214, 70, 11, 0, 12, 42, 45],# 39+3
    ['박동원', 215, 181, 55, 7, 0, 13, 45, 30], # 27+2
    ['오지환', 202, 175, 39, 10, 0, 6, 52, 18], # 15+3
    ['문성주', 191, 166, 44, 6, 1, 0, 24, 26],  # 22+0
    ['송찬의', 145, 128, 30, 9, 1, 3, 41, 15],  # 8+7
    ['신민재', 158, 132, 32, 1, 0, 0, 16, 24],  # 23+1
]


columns =['Player', 'PA','AB', '1B', '2B', '3B', 'HR', 'SO', 'BB']
batter = pd.DataFrame(data, columns=columns)

batter["1B"] = batter["1B"] - batter["2B"] - batter["3B"] - batter["HR"]
batter["OUT"] = batter["AB"] - (batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"])
batter["FLY"] = (batter["OUT"] / 2).astype(int)
batter["POPUP"] = 0
batter["GROUND"] = batter["OUT"] - batter["FLY"]

batter = batter.drop('AB',axis=1)
batter["PA"] = batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"] + batter["BB"] + batter["OUT"]

batter['OBP'] = ((batter['1B'] + batter['2B'] + batter['3B'] + batter['HR']+ batter['BB']) / batter['PA']).round(3)
batter['SLG'] = ((batter['1B'] + 2 * batter['2B'] + 3 * batter['3B'] + 4 * batter['HR']) / (batter['PA'] - batter['BB'] )).round(3)
batter['OPS'] = (batter['OBP'] + batter['SLG']).round(3)


LG_batter = batting_order(batter)
LG_batter = LG_batter.drop(["OBP","SLG","OPS"],axis=1)
LG_batter.index = range(1,10)

In [4]:
# TBF, H, 2B, 3B, HR, SO, BB+HBP

data =  [
    ['치리노스',292,62,9,0,3,69,18],
    ['에르난데스',124,23,6,0,5,34,10],
    ['임찬규',307,69,18,1,3,55,24],
    ['손주영',266,60,6,1,5,64,27],
    ['송승기',259,49,9,0,5,62,21]
    
    
    
]
columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,6)
LG_starter = pitcher.drop(["OBP","SLG","OPS"],axis=1)


data = [
    ['김진성',129,21,1,1,2,29,15],
    ['박명근',106,22,0,1,4,23,12],
    ['김영우',101,21,6,0,0,28,14],
    ['백승현',96,15,0,0,1,19,20],
    ['이우찬',81,11,2,0,0,19,14],
    ['이지강',97,22,2,1,3,16,11],
    ['장현식',60,12,4,0,1,14,5],
    ['김강률',51,7,1,0,0,9,9]
]
columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)
pitcher['OPS'] = pitcher['OPS'] - pitcher['PA']/1000 # 이닝수 조정

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,9)
LG_bullpen = pitcher.drop(["OBP","SLG","OPS"],axis=1)

In [5]:
# HH

# [이름, PA, AB, H, 2B, 3B, HR, SO, BB+HBP]
data = [
    ['플로리얼', 270, 246, 67, 17, 2, 8, 62, 25],# 23+0
    ['문현빈', 236, 212, 67, 14, 1, 8, 37, 21],  # 16+1
    ['노시환', 262, 230, 53, 7, 1, 11, 58, 37],  # 26+5
    ['채은성', 240, 217, 61, 15, 1, 9, 52, 22],  # 16+5
    ['이진영', 177, 152, 43, 8, 0, 5, 40, 23],   # 20+2
    ['황영묵', 174, 156, 37, 9, 0, 1, 22, 16],   # 16+0
    ['최인호', 82, 73, 18, 4, 0, 1, 14, 8],      # 7+1
    ['최재훈', 133, 95, 26, 5, 0, 0, 14, 30],   # 20+10
    ['심우준', 101, 94, 16, 3, 1, 1, 21, 4],     # 3+0
    
]


columns =['Player', 'PA','AB', '1B', '2B', '3B', 'HR', 'SO', 'BB']
batter = pd.DataFrame(data, columns=columns)

batter["1B"] = batter["1B"] - batter["2B"] - batter["3B"] - batter["HR"]
batter["OUT"] = batter["AB"] - (batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"])
batter["FLY"] = (batter["OUT"] / 2).astype(int)
batter["POPUP"] = 0
batter["GROUND"] = batter["OUT"] - batter["FLY"]

batter = batter.drop('AB',axis=1)
batter["PA"] = batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"] + batter["BB"] + batter["OUT"]


batter['OBP'] = ((batter['1B'] + batter['2B'] + batter['3B'] + batter['HR']+ batter['BB']) / batter['PA']).round(3)
batter['SLG'] = ((batter['1B'] + 2 * batter['2B'] + 3 * batter['3B'] + 4 * batter['HR']) / (batter['PA'] - batter['BB'] )).round(3)
batter['OPS'] = (batter['OBP'] + batter['SLG']).round(3)


HH_batter = batting_order(batter)
HH_batter = HH_batter.drop(["OBP","SLG","OPS"],axis=1)
HH_batter.index = range(1,10)

In [6]:
# TBF, H, 2B, 3B, HR, SO, BB+HBP

data = [
    ['와이스',328,64,10,0,9,90,25],
    ['폰세',319,52,7,0,4,112,21],
    ['류현진',290,71,7,1,6,57,19],
    ['문동주',211,44,8,2,2,56,16],
    ['엄상백',179,51,8,1,6,29,22],
]

columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,6)
HH_starter = pitcher.drop(["OBP","SLG","OPS"],axis=1)

data = [
    ['박상원',124,24,7,0,2,24,15],
    ['김서현',118,20,1,2,1,32,14],
    ['한승혁',120,24,3,1,2,22,13],
    ['정우주',101,13,4,0,4,31,16],
    ['조동욱',118,30,7,0,1,14,16],
    ['김범수',64,11,2,0,0,18,7],
    ['김종수',100,20,7,0,0,22,18],
    ['주현상',62,18,2,0,2,11,5]
]
columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)
pitcher['OPS'] = pitcher['OPS'] - pitcher['PA']/1000 # 이닝수 조정

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,9)
HH_bullpen = pitcher.drop(["OBP","SLG","OPS"],axis=1)

In [7]:
# LT

data = [
    ['장두성', 152, 132, 39, 3, 0, 0, 30, 12],
    ['고승민', 230, 204, 62, 11, 1, 2, 33, 22],
    ['레이예스', 279, 255, 85, 23, 0, 7, 33, 23],
    ['전준우', 260, 221, 65, 16, 1, 5, 38, 33],
    ['윤동희', 217, 184, 55, 10, 0, 4, 36, 29],
    ['전민재', 178, 158, 58, 10, 0, 3, 29, 12],
    ['나승엽', 238, 199, 49, 9, 1, 7, 35, 39],
    ['손호영', 178, 156, 40, 3, 0, 2, 33, 15],
    ['유강남', 159, 129, 38, 9, 0, 4, 36, 26],
]




columns =['Player', 'PA','AB', '1B', '2B', '3B', 'HR', 'SO', 'BB']
batter = pd.DataFrame(data, columns=columns)

batter["1B"] = batter["1B"] - batter["2B"] - batter["3B"] - batter["HR"]
batter["OUT"] = batter["AB"] - (batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"])
batter["FLY"] = (batter["OUT"] / 2).astype(int)
batter["POPUP"] = 0
batter["GROUND"] = batter["OUT"] - batter["FLY"]

batter = batter.drop('AB',axis=1)
batter["PA"] = batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"] + batter["BB"] + batter["OUT"]



batter['OBP'] = ((batter['1B'] + batter['2B'] + batter['3B'] + batter['HR']+ batter['BB']) / batter['PA']).round(3)
batter['SLG'] = ((batter['1B'] + 2 * batter['2B'] + 3 * batter['3B'] + 4 * batter['HR']) / (batter['PA'] - batter['BB'] )).round(3)
batter['OPS'] = (batter['OBP'] + batter['SLG']).round(3)


LT_batter = batting_order(batter)
LT_batter = LT_batter.drop(["OBP","SLG","OPS"],axis=1)
LT_batter.index = range(1,10)

In [8]:
# TBF, H, 2B, 3B, HR, SO, BB+HBP

data = [
    ['데이비슨',312,70,13,1,5,68,32],
    ['박세웅',340,71,17,1,4,85,37],
    ['나균안',259,64,8,0,6,41,32],
    ['이민석',113,27,4,0,4,16,11],
    ['감보아',47,7,0,0,0,15,4],
]

columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,6)
LT_starter = pitcher.drop(["OBP","SLG","OPS"],axis=1)

data = [
    ['김상수',141,36,7,0,4,24,16],
    ['정현수',107,18,6,0,1,28,15],
    ['송재영',86,13,4,0,2,22,17],
    ['정철원',132,30,6,1,2,25,15],
    ['김강현',144,27,5,0,3,22,16],
    ['김원중',111,17,3,1,2,33,15],
    ['박진',136,36,6,0,5,15,9],
    ['김진욱',115,31,5,0,7,23,15]
]
columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)
pitcher['OPS'] = pitcher['OPS'] - pitcher['PA']/1000 # 이닝수 조정

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,9)
LT_bullpen = pitcher.drop(["OBP","SLG","OPS"],axis=1)

In [9]:
# SS

data = [
    ['김성윤', 212, 179, 64, 12, 3, 2, 28, 26],
    ['김지찬', 142, 124, 37, 5, 1, 0, 14, 15],
    ['구자욱', 254, 219, 55, 15, 0, 9, 43, 33],
    ['디아즈', 261, 234, 68, 12, 0, 22, 50, 26],
    ['강민호', 211, 187, 53, 12, 0, 3, 31, 23],
    ['김영웅', 207, 188, 48, 9, 1, 8, 61, 19],
    ['박병호', 151, 125, 25, 3, 0, 9, 46, 24],
    ['류지혁', 210, 179, 51, 7, 0, 0, 33, 22],
    ['이재현', 246, 198, 46, 8, 0, 6, 56, 42],    
]




columns =['Player', 'PA','AB', '1B', '2B', '3B', 'HR', 'SO', 'BB']
batter = pd.DataFrame(data, columns=columns)

batter["1B"] = batter["1B"] - batter["2B"] - batter["3B"] - batter["HR"]
batter["OUT"] = batter["AB"] - (batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"])
batter["FLY"] = (batter["OUT"] / 2).astype(int)
batter["POPUP"] = 0
batter["GROUND"] = batter["OUT"] - batter["FLY"]

batter = batter.drop('AB',axis=1)
batter["PA"] = batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"] + batter["BB"] + batter["OUT"]

batter['OBP'] = ((batter['1B'] + batter['2B'] + batter['3B'] + batter['HR']+ batter['BB']) / batter['PA']).round(3)
batter['SLG'] = ((batter['1B'] + 2 * batter['2B'] + 3 * batter['3B'] + 4 * batter['HR']) / (batter['PA'] - batter['BB'] )).round(3)
batter['OPS'] = (batter['OBP'] + batter['SLG']).round(3)


SS_batter = batting_order(batter)
SS_batter = SS_batter.drop(["OBP","SLG","OPS"],axis=1)
SS_batter.index = range(1,10)

In [10]:
# TBF, H, 2B, 3B, HR, SO, BB+HBP

data = [
    ['후라도',342,82,5,1,8,63,20],
    ['원태인',267,59,17,1,5,51,10],
    ['이승현',222,55,13,1,5,32,25],
    ['최원태',254,57,8,0,5,57,32],
    ['레예스',199,45,6,1,4,32,13],
]

columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,6)
SS_starter = pitcher.drop(["OBP","SLG","OPS"],axis=1)

data = [
    ['김태훈',120,25,6,0,1,32,10],
    ['이호성',133,24,3,2,2,39,16],
    ['배찬승',113,28,5,0,2,24,16],
    ['백정현',128,19,6,0,1,31,9],
    ['김재윤',103,25,4,1,5,18,5],
    ['이승민',90,22,6,0,2,22,7],
    ['임창민',50,12,2,0,4,4,8],
    ['양창섭',66,19,3,0,1,11,8]
]
columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)
pitcher['OPS'] = pitcher['OPS'] - pitcher['PA']/1000 # 이닝수 조정

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,9)
SS_bullpen = pitcher.drop(["OBP","SLG","OPS"],axis=1)

In [11]:
# SSG
# [이름, PA, AB, H, 2B, 3B, HR, SO, BB+HBP]
data = [
    ['최지훈',258,233,69,9,3,2,44,21 ],
    ['에레디아',70,60,17,2,0,1,9,11 ],
    ['최정',119,97,20,3,0,9,28,23 ],
    ['고명준',215,200,53,10,1,7,43,11],
    ['한유섬',225,200,52,14,0,4,55,26 ],
    ['박성한',232,202,44,10,0,3,49,39 ],
    ['최준우',134,101,21,0,0,3,32,27 ],
    ['조형우',116,107,27,1,0,2,18,7 ],
    ['정준재',204,176,38,5,1,0,49,23 ],
]





columns =['Player', 'PA','AB', '1B', '2B', '3B', 'HR', 'SO', 'BB']
batter = pd.DataFrame(data, columns=columns)

batter["1B"] = batter["1B"] - batter["2B"] - batter["3B"] - batter["HR"]
batter["OUT"] = batter["AB"] - (batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"])
batter["FLY"] = (batter["OUT"] / 2).astype(int)
batter["POPUP"] = 0
batter["GROUND"] = batter["OUT"] - batter["FLY"]

batter = batter.drop('AB',axis=1)
batter["PA"] = batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"] + batter["BB"] + batter["OUT"]

batter['OBP'] = ((batter['1B'] + batter['2B'] + batter['3B'] + batter['HR']+ batter['BB']) / batter['PA']).round(3)
batter['SLG'] = ((batter['1B'] + 2 * batter['2B'] + 3 * batter['3B'] + 4 * batter['HR']) / (batter['PA'] - batter['BB'] )).round(3)
batter['OPS'] = (batter['OBP'] + batter['SLG']).round(3)


SSG_batter = batting_order(batter)
SSG_batter = SSG_batter.drop(["OBP","SLG","OPS"],axis=1)
SSG_batter.index = range(1,10)

In [12]:
# TBF, H, 2B, 3B, HR, SO, BB+HBP

data = [
    ['김광현',309,73,13,1,6,71,26],
    ['앤더슨',279,48,8,1,4,98,24],
    ['문승원',192,43,11,2,5,21,15],
    ['화이트',211,38,7,1,4,55,16],
    ['송영진',143,41,2,0,3,25,17],
]

columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,6)
SSG_starter = pitcher.drop(["OBP","SLG","OPS"],axis=1)

data = [
    ['노경은',139,29,6,0,1,26,12],
    ['김민', 114,29,5,0,3,25,10],
    ['이로운',130,26,2,1,0,30,14],
    ['조병현',108,21,4,0,4,28,4],
    ['한두솔',97,26,4,0,0,18,16],
    ['김건우',144,26,4,0,1,33,26],
    ['박시후',91,18,0,0,4,16,11],
    ['최민준',84,13,1,0,3,16,13]
]
columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)
pitcher['OPS'] = pitcher['OPS'] - pitcher['PA']/1000 # 이닝수 조정

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,9)
SSG_bullpen = pitcher.drop(["OBP","SLG","OPS"],axis=1)

In [13]:
# 두산
# [이름, PA, AB, H, 2B, 3B, HR, SO, BB+HBP]
data = [
    ['정수빈',254,212,55,5,0,3,22,39 ],
    ['케이브',239,220,65,10,1,4,51,17 ],
    ['양의지',240,204,63,12,0,9,32,35 ],
    ['김재환',232,200,48,6,1,7,54,34 ],
    ['양석환',232,204,53,13,0,6,60,25 ],
    ['김인태',91,76,21,5,0,2,22,15 ],
    ['오명진',150,135,37,7,2,1,31,14 ],
    ['강승호',223,203,44,12,2,3,71,19 ],
    ['임종성',69,64,17,3,0,1,18,4 ],

]





columns =['Player', 'PA','AB', '1B', '2B', '3B', 'HR', 'SO', 'BB']
batter = pd.DataFrame(data, columns=columns)

batter["1B"] = batter["1B"] - batter["2B"] - batter["3B"] - batter["HR"]
batter["OUT"] = batter["AB"] - (batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"])
batter["FLY"] = (batter["OUT"] / 2).astype(int)
batter["POPUP"] = 0
batter["GROUND"] = batter["OUT"] - batter["FLY"]

batter = batter.drop('AB',axis=1)
batter["PA"] = batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"] + batter["BB"] + batter["OUT"]

batter['OBP'] = ((batter['1B'] + batter['2B'] + batter['3B'] + batter['HR']+ batter['BB']) / batter['PA']).round(3)
batter['SLG'] = ((batter['1B'] + 2 * batter['2B'] + 3 * batter['3B'] + 4 * batter['HR']) / (batter['PA'] - batter['BB'] )).round(3)
batter['OPS'] = (batter['OBP'] + batter['SLG']).round(3)


DS_batter = batting_order(batter)
DS_batter = DS_batter.drop(["OBP","SLG","OPS"],axis=1)
DS_batter.index = range(1,10)

In [14]:
# TBF, H, 2B, 3B, HR, SO, BB+HBP

data = [
    ['최원준',291,69,12,1,11,40,25],
    ['잭로그',291,54,10,1,1,72,25],
    ['콜어빈',292,53,8,2,2,54,46],
    ['최승용',251,56,9,0,5,42,25],
    ['김유성',64,13,1,0,2,14,14],
]

columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,6)
DS_starter = pitcher.drop(["OBP","SLG","OPS"],axis=1)

data = [
    ['박치국',125,29,7,0,1,30,11],
    ['이영하',131,27,5,0,2,32,18],
    ['김택연',120,20,2,0,3,38,10],
    ['최지강',84,20,2,0,2,26,4],
    ['김호준',54,14,3,0,2,7,7],
    ['박신지',104,25,8,0,1,11,10],
    ['고효준',50,14,1,0,2,7,6],
    ['홍민규',126,30,6,0,4,17,12]
]
columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)
pitcher['OPS'] = pitcher['OPS'] - pitcher['PA']/1000 # 이닝수 조정

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,9)
DS_bullpen = pitcher.drop(["OBP","SLG","OPS"],axis=1)

In [15]:
# KIA
# [이름, PA, AB, H, 2B, 3B, HR, SO, BB+HBP]
data = [
    ['박찬호', 224, 187, 49, 8, 0, 1, 23, 29],
    ['김선빈', 125, 106, 31, 10, 0, 0, 11, 17],
    ['김도영', 111, 100, 33, 9, 0, 7, 18, 10],
    ['최형우', 235, 199, 67, 18, 1, 10, 34, 37],
    ['위즈덤', 167, 140, 36, 9, 0, 10, 43, 26],
    ['나성범', 110, 93, 21, 5, 0, 4, 21, 17],
    ['최원준', 155, 142, 31, 3, 0, 4, 27, 13],
    ['이우성', 168, 147, 34, 10, 1, 2, 35, 20],
    ['김태군', 101, 89, 20, 6, 0, 1, 5, 8],
]





columns =['Player', 'PA','AB', '1B', '2B', '3B', 'HR', 'SO', 'BB']
batter = pd.DataFrame(data, columns=columns)

batter["1B"] = batter["1B"] - batter["2B"] - batter["3B"] - batter["HR"]
batter["OUT"] = batter["AB"] - (batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"])
batter["FLY"] = (batter["OUT"] / 2).astype(int)
batter["POPUP"] = 0
batter["GROUND"] = batter["OUT"] - batter["FLY"]

batter = batter.drop('AB',axis=1)
batter["PA"] = batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"] + batter["BB"] + batter["OUT"]

batter['OBP'] = ((batter['1B'] + batter['2B'] + batter['3B'] + batter['HR']+ batter['BB']) / batter['PA']).round(3)
batter['SLG'] = ((batter['1B'] + 2 * batter['2B'] + 3 * batter['3B'] + 4 * batter['HR']) / (batter['PA'] - batter['BB'] )).round(3)
batter['OPS'] = (batter['OBP'] + batter['SLG']).round(3)


KIA_batter = batting_order(batter)
KIA_batter = KIA_batter.drop(["OBP","SLG","OPS"],axis=1)
KIA_batter.index = range(1,10)

In [16]:
# TBF, H, 2B, 3B, HR, SO, BB+HBP

data = [
    ['네일',322,70,12,1,2,71,29],
    ['김도현',296,70,15,1,5,43,23],
    ['양현종',287,74,12,3,3,51,29],
    ['올러',287,54,15,1,3,77,19],
    ['윤영철',120,27,2,0,3,19,22],
]

columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,6)
KIA_starter = pitcher.drop(["OBP","SLG","OPS"],axis=1)

data = [
    ['전상현',122,29,6,0,1,17,12],
    ['조상우',128,28,5,0,3,32,19],
    ['이준영',78,19,5,0,2,20,6],
    ['정해영',121,29,5,0,1,36,9],
    ['최지민',90,17,2,0,1,14,22],
    ['황동하',127,30,12,1,3,23,13],
    ['김건국',70,18,2,0,2,11,10],
    ['윤중현',44,9,4,0,2,7,5]
]
columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)
pitcher['OPS'] = pitcher['OPS'] - pitcher['PA']/1000 # 이닝수 조정

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,9)
KIA_bullpen = pitcher.drop(["OBP","SLG","OPS"],axis=1)

In [17]:
# KT
# [이름, PA, AB, H, 2B, 3B, HR, SO, BB+HBP]
data = [
    ['배정대', 158, 140, 28, 7, 2, 1, 38, 11],
    ['김민혁', 237, 217, 64, 7, 1, 0, 18, 18],
    ['안현민', 140, 123, 40, 7, 3, 10, 22, 17],
    ['로하스', 269, 227, 60, 13, 0, 8, 46, 41],
    ['장성우', 221, 183, 45, 8, 0, 5, 42, 33],
    ['문상철', 147, 126, 27, 5, 1, 1, 32, 18],
    ['허경민', 157, 138, 36, 3, 0, 1, 18, 19],
    ['김상수', 135, 114, 24, 5, 0, 1, 10, 22],
    ['권동진', 148, 124, 31, 4, 1, 0, 36, 19],
    
    
    
    
]

columns =['Player', 'PA','AB', '1B', '2B', '3B', 'HR', 'SO', 'BB']
batter = pd.DataFrame(data, columns=columns)

batter["1B"] = batter["1B"] - batter["2B"] - batter["3B"] - batter["HR"]
batter["OUT"] = batter["AB"] - (batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"])
batter["FLY"] = (batter["OUT"] / 2).astype(int)
batter["POPUP"] = 0
batter["GROUND"] = batter["OUT"] - batter["FLY"]

batter = batter.drop('AB',axis=1)
batter["PA"] = batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"] + batter["BB"] + batter["OUT"]

batter['OBP'] = ((batter['1B'] + batter['2B'] + batter['3B'] + batter['HR']+ batter['BB']) / batter['PA']).round(3)
batter['SLG'] = ((batter['1B'] + 2 * batter['2B'] + 3 * batter['3B'] + 4 * batter['HR']) / (batter['PA'] - batter['BB'] )).round(3)
batter['OPS'] = (batter['OBP'] + batter['SLG']).round(3)


KT_batter = batting_order(batter)
KT_batter = KT_batter.drop(["OBP","SLG","OPS"],axis=1)
KT_batter.index = range(1,10)

In [18]:
# TBF, H, 2B, 3B, HR, SO, BB+HBP

data = [
    ['쿠에바스',321,80,10,1,12,44,36],
    ['오원석',279,56,8,0,5,60,32],
    ['헤이수스',285,57,8,1,7,77,21],
    ['고영표',289,74,15,0,2,70,17],
    ['소형준',245,50,11,1,2,63,17],
]

columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,6)
KT_starter = pitcher.drop(["OBP","SLG","OPS"],axis=1)

data = [
    ['김민수',135,35,6,2,3,20,8],
    ['박영현',135,27,6,0,2,36,15],
    ['손동현',124,23,5,1,2,29,8],
    ['원상현',126,20,5,0,3,26,16],
    ['우규민',89,20,5,0,0,14,3],
    ['문용익',65,11,4,0,3,16,11],
    ['주권',50,13,1,0,0,6,4],
    ['최동환',56,20,2,0,2,6,8]
]
columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)
pitcher['OPS'] = pitcher['OPS'] - pitcher['PA']/1000 # 이닝수 조정

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,9)
KT_bullpen = pitcher.drop(["OBP","SLG","OPS"],axis=1)

In [19]:
# 키움
# [이름, PA, AB, H, 2B, 3B, HR, SO, BB+HBP]
data = [
    ['송성문',279,250,70,17,0,9,39,29 ],
    ['최주환',254,228,64,14,1,4,33,22 ],
    ['이주형',189,165,41,7,0,6,51,24 ],
    ['카디네스',222,189,45,8,1,5,43,32 ],
    ['이형종',79,68,14,3,0,2,22,9 ],
    ['임병욱',85,81,19,3,1,0,20,3 ],
    ['김태진',181,165,42,9,1,3,33,14 ],
    ['오선진',99,87,21,4,0,1,26,9 ],
    ['김재현',95,91,19,4,0,0,22,3 ],
]





columns =['Player', 'PA','AB', '1B', '2B', '3B', 'HR', 'SO', 'BB']
batter = pd.DataFrame(data, columns=columns)

batter["1B"] = batter["1B"] - batter["2B"] - batter["3B"] - batter["HR"]
batter["OUT"] = batter["AB"] - (batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"])
batter["FLY"] = (batter["OUT"] / 2).astype(int)
batter["POPUP"] = 0
batter["GROUND"] = batter["OUT"] - batter["FLY"]

batter = batter.drop('AB',axis=1)
batter["PA"] = batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"] + batter["BB"] + batter["OUT"]

batter['OBP'] = ((batter['1B'] + batter['2B'] + batter['3B'] + batter['HR']+ batter['BB']) / batter['PA']).round(3)
batter['SLG'] = ((batter['1B'] + 2 * batter['2B'] + 3 * batter['3B'] + 4 * batter['HR']) / (batter['PA'] - batter['BB'] )).round(3)
batter['OPS'] = (batter['OBP'] + batter['SLG']).round(3)


KW_batter = batting_order(batter)
KW_batter = KW_batter.drop(["OBP","SLG","OPS"],axis=1)
KW_batter.index = range(1,10)

In [20]:
# TBF, H, 2B, 3B, HR, SO, BB+HBP

data = [
    ['하영민',323,83,10,1,6,70,28],
    ['김윤하',280,79,14,1,8,32,31],
    ['로젠버그',299,60,12,1,5,77,26],
    ['김선기',207,50,9,0,3,26,35],
    ['알칸타라',26,6,3,0,0,4,2],
]

columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,6)
KW_starter = pitcher.drop(["OBP","SLG","OPS"],axis=1)

data = [
    ['김성민',94,20,3,3,1,13,15],
    ['원종현',110,33,4,2,4,14,10],
    ['오석주',118,36,5,1,2,19,13],
    ['이강준',95,22,4,0,2,20,13],
    ['박윤성',80,21,5,0,3,17,7],
    ['이준우',66,17,4,1,1,13,8],
    ['주승우',84,15,1,0,5,17,6],
    ['양지율',69,18,1,2,3,10,6]
    
]
columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)
pitcher['OPS'] = pitcher['OPS'] - pitcher['PA']/1000 # 이닝수 조정

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,9)
KW_bullpen = pitcher.drop(["OBP","SLG","OPS"],axis=1)

In [21]:
# NC
# [이름, PA, AB, H, 2B, 3B, HR, SO, BB+HBP]
data = [
    ['박민우',208,180,53,11,5,0,32,29],
    ['김주원',240,206,47,6,2,4,43,30],
    ['박건우',134,115,35,8,0,1,18,17],
    ['데이비슨',140,122,39,6,0,10,39,13],
    ['손아섭',168,150,48,9,3,0,22,18],
    ['권희동',200,157,41,12,0,3,33,40],
    ['서호철',115,102,30,4,0,1,17,6],
    ['천재환',139,122,32,3,2,3,27,13],
    ['김형준',149,133,32,3,1,11,46,14]
]

columns =['Player', 'PA','AB', '1B', '2B', '3B', 'HR', 'SO', 'BB']
batter = pd.DataFrame(data, columns=columns)

batter["1B"] = batter["1B"] - batter["2B"] - batter["3B"] - batter["HR"]
batter["OUT"] = batter["AB"] - (batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"])
batter["FLY"] = (batter["OUT"] / 2).astype(int)
batter["POPUP"] = 0
batter["GROUND"] = batter["OUT"] - batter["FLY"]

batter = batter.drop('AB',axis=1)
batter["PA"] = batter["1B"] + batter["2B"] + batter["3B"] + batter["HR"] + batter["SO"] + batter["BB"] + batter["OUT"]

batter['OBP'] = ((batter['1B'] + batter['2B'] + batter['3B'] + batter['HR']+ batter['BB']) / batter['PA']).round(3)
batter['SLG'] = ((batter['1B'] + 2 * batter['2B'] + 3 * batter['3B'] + 4 * batter['HR']) / (batter['PA'] - batter['BB'] )).round(3)
batter['OPS'] = (batter['OBP'] + batter['SLG']).round(3)


NC_batter = batting_order(batter)
NC_batter = NC_batter.drop(["OBP","SLG","OPS"],axis=1)
NC_batter.index = range(1,10)

In [22]:
# TBF, H, 2B, 3B, HR, SO, BB+HBP

data = [
    ['라일리',317,57,11,0,8,92,33],
    ['로건',321,67,14,0,2,51,40],
    ['신민혁',206,48,10,1,5,31,12],
    ['목지훈',160,37,6,0,6,29,28],
    ['최성영',121,33,5,2,3,18,13],
]

columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,6)
NC_starter = pitcher.drop(["OBP","SLG","OPS"],axis=1)

data = [
    ['김진호',105,15,1,0,2,24,18],
    ['손주환',116,19,0,0,2,22,11],
    ['류진욱',106,23,5,1,5,20,13],
    ['전사민',123,35,2,2,2,13,14],
    ['김재열',104,30,6,1,4,14,15],
    ['배재환',84,13,2,0,0,19,12],
    ['한재승',84,18,2,0,1,16,19],
    ['김시훈',58,17,3,0,2,7,6]
]
columns =['Player', 'PA', '1B', '2B', '3B', 'HR', 'SO', 'BB']
pitcher = pd.DataFrame(data, columns=columns)

pitcher["1B"] = pitcher["1B"] - pitcher["2B"] - pitcher["3B"] - pitcher["HR"]
pitcher["OUT"] = pitcher["PA"] - (pitcher["1B"] + pitcher["2B"] + pitcher["3B"] + pitcher["HR"] + pitcher["SO"] + pitcher["BB"])
pitcher["FLY"] = (pitcher["OUT"] / 2).astype(int)
pitcher["POPUP"] = 0
pitcher["GROUND"] = pitcher["OUT"] - pitcher["FLY"]

pitcher['OBP'] = ((pitcher['1B'] + pitcher['2B'] + pitcher['3B'] + pitcher['HR']+ pitcher['BB']) / pitcher['PA']).round(3)
pitcher['SLG'] = ((pitcher['1B'] + 2 * pitcher['2B'] + 3 * pitcher['3B'] + 4 * pitcher['HR']) / (pitcher['PA'] - pitcher['BB'] )).round(3)
pitcher['OPS'] = (pitcher['OBP'] + pitcher['SLG']).round(3)
pitcher['OPS'] = pitcher['OPS'] - pitcher['PA']/1000 # 이닝수 조정

pitcher = pitcher.sort_values(by="OPS", ascending=True).reset_index(drop=True)
pitcher.index = range(1,9)
NC_bullpen = pitcher.drop(["OBP","SLG","OPS"],axis=1)

In [23]:
LG_roster = [LG_batter, LG_starter, LG_bullpen]
HH_roster = [HH_batter, HH_starter, HH_bullpen]
SS_roster = [SS_batter, SS_starter, SS_bullpen]
SSG_roster = [SSG_batter, SSG_starter, SSG_bullpen]
DS_roster = [DS_batter, DS_starter, DS_bullpen]
LT_roster = [LT_batter, LT_starter, LT_bullpen]
KW_roster = [KW_batter, KW_starter, KW_bullpen]
KIA_roster = [KIA_batter, KIA_starter, KIA_bullpen]
KT_roster = [KT_batter, KT_starter, KT_bullpen]
NC_roster = [NC_batter, NC_starter, NC_bullpen]


In [24]:
KW_roster[0]

,Player,PA,1B,2B,3B,HR,SO,BB,OUT,FLY,POPUP,GROUND
1,카디네스,221,31,8,1,5,43,32,101,50,0,51
2,김태진,179,29,9,1,3,33,14,90,45,0,45
3,최주환,250,45,14,1,4,33,22,131,65,0,66
4,송성문,279,44,17,0,9,39,29,141,70,0,71
5,이주형,189,28,7,0,6,51,24,73,36,0,37
6,이형종,77,9,3,0,2,22,9,32,16,0,16
7,오선진,96,16,4,0,1,26,9,40,20,0,20
8,임병욱,84,15,3,1,0,20,3,42,21,0,21
9,김재현,94,15,4,0,0,22,3,50,25,0,25


# Single_AB :  임찬규 VS 양의지

In [25]:
print( LG_starter.loc[3] )

Player    임찬규
PA        307
1B         47
2B         18
3B          1
HR          3
SO         55
BB         24
OUT       159
FLY        79
POPUP       0
GROUND     80
Name: 3, dtype: object


In [26]:
print( DS_batter.loc[4] )

Player    양의지
PA        239
1B         42
2B         12
3B          0
HR          9
SO         32
BB         35
OUT       109
FLY        54
POPUP       0
GROUND     55
Name: 4, dtype: object


In [27]:
# 임찬규 + 양의지
print( LG_starter.loc[3][1:] + DS_batter.loc[4][1:]  )

PA        546
1B         89
2B         30
3B          1
HR         12
SO         87
BB         59
OUT       268
FLY       133
POPUP       0
GROUND    135
dtype: object


In [28]:
# 임찬규 + 양의지 10타석 결과
for i in range(10):
    print( single_AB(LG_starter.loc[3],DS_batter.loc[4],ratio_option = False) )

SO
SO
SO
1B
GROUND
1B
SO
1B
FLY
2B


In [29]:
# 임찬규 + 양의지 10타석 결과 (민재형 제안대로 1대1로 비율 맞추기)
for i in range(10):
    print( single_AB(LG_starter.loc[3],DS_batter.loc[4],ratio_option = True) )

2B
BB
GROUND
FLY
GROUND
FLY
2B
1B
GROUND
FLY


# 타석 결과가 뜬공일 때 무사에서 누상 변화

In [48]:
# 누상 변화가 다 똑같음 (플라이는 변수 적음)

res = "FLY"
for i in range(10):
    print( base_status(res,"000",0,0) )
    

('000', 1, 0, 'F9')
('000', 1, 0, 'F7')
('000', 1, 0, 'F9')
('000', 1, 0, 'F7')
('000', 1, 0, 'F8')
('000', 1, 0, 'F8')
('000', 1, 0, 'F8')
('000', 1, 0, 'F9')
('000', 1, 0, 'F8')
('000', 1, 0, 'F7')


# 타석 결과가 땅볼일 때 1사 1루에서 누상 변화

In [49]:
# 누상 변화가 다름 (에러, 병살타, 진루타 등 다양한 상황 야기)

res = "GROUND"
for i in range(10):
    print( base_status(res,"100",1,0) )

('010', 2, 0, 'G5')
('000', 3, 0, 'DP')
('010', 2, 0, 'G5')
('100', 2, 0, 'G6')
('010', 2, 0, 'G6')
('010', 2, 0, 'G6')
('110', 1, 0, 'E4')
('010', 2, 0, 'G6')
('010', 2, 0, 'G1')
('100', 2, 0, 'G6')


# 타석 결과가 1루타일 때 2사 1,2루에서 누상 변화

In [50]:
# 누상 변화와 득점이 다름 (1점 or 0점)

res = "1B"
for i in range(10):
    print( base_status(res,"110",2,0) )

('110', 2, 1, '1B')
('111', 2, 0, '1B')
('110', 2, 1, '1B')
('110', 2, 1, '1B')
('110', 2, 1, '1B')
('101', 2, 1, '1B')
('110', 2, 1, '1B')
('110', 2, 1, '1B')
('110', 2, 1, '1B')
('110', 2, 1, '1B')


# 타석 결과가 플라이일 때 무사 3루에서 누상 변화

In [51]:
# 누상 변화와 득점이 다름 (1점 or 0점)

res = "FLY"
for i in range(10):
    print( base_status(res,"001",0,0) )

('000', 1, 1, 'SAC')
('000', 1, 1, 'SAC')
('000', 1, 1, 'SAC')
('000', 1, 1, 'SAC')
('001', 1, 0, 'F8')
('000', 1, 1, 'SAC')
('000', 1, 1, 'SAC')
('000', 1, 1, 'SAC')
('000', 1, 1, 'SAC')
('000', 1, 1, 'SAC')


# 1이닝 시뮬레이션 / 4번 양의지부터 

In [34]:
one_innging(DS_batter,LG_starter,LG_bullpen,4,3,0,print_record = True,return_res_record = True)

임찬규 / PITCH COUNT :  0
4 .  양의지 : G5 / BASE STATUS : 000 / OUT COUNT : 1
--------------------------------------------------------------
임찬규 / PITCH COUNT :  4
5 .  양석환 : G4 / BASE STATUS : 000 / OUT COUNT : 2
--------------------------------------------------------------
임찬규 / PITCH COUNT :  11
6 .  케이브 : BB / BASE STATUS : 100 / OUT COUNT : 2
--------------------------------------------------------------
임찬규 / PITCH COUNT :  15
7 .  오명진 : G5 / BASE STATUS : 100 / OUT COUNT : 3
--------------------------------------------------------------
SCORE :  0


(0, 19, 3, 8, ['OUT', 'OUT', 'BB', 'OUT'], [0, 0, 0, 0])

# 1이닝 시뮬레이션 / 1번 김현수부터 

In [35]:
one_innging(LG_batter,DS_starter,DS_bullpen,1,4,0,print_record = True,return_res_record = True)

최원준 / PITCH COUNT :  0
1 .  김현수 : HR / BASE STATUS : 000 / OUT COUNT : 0
--------------------------------------------------------------
최원준 / PITCH COUNT :  1
2 .  문성주 : HR / BASE STATUS : 000 / OUT COUNT : 0
--------------------------------------------------------------
최원준 / PITCH COUNT :  8
3 .  문보경 : F8 / BASE STATUS : 000 / OUT COUNT : 1
--------------------------------------------------------------
최원준 / PITCH COUNT :  11
4 .  오스틴 : F9 / BASE STATUS : 000 / OUT COUNT : 2
--------------------------------------------------------------
최원준 / PITCH COUNT :  16
5 .  박동원 : BB / BASE STATUS : 100 / OUT COUNT : 2
--------------------------------------------------------------
최원준 / PITCH COUNT :  23
6 .  송찬의 : F8 / BASE STATUS : 100 / OUT COUNT : 3
--------------------------------------------------------------
SCORE :  2


(2, 24, 4, 7, ['HR', 'HR', 'OUT', 'OUT', 'BB', 'OUT'], [1, 1, 0, 0, 0, 0])

In [36]:
single_game("LG","DS",LG_roster,DS_roster,3,4,"LG",print_record = True,print_HBE = True)

 
 
[ TOP of 1 INNING ]
최원준 / PITCH COUNT :  0
1 .  김현수 : F7 / BASE STATUS : 000 / OUT COUNT : 1
--------------------------------------------------------------
최원준 / PITCH COUNT :  3
2 .  문성주 : 2B / BASE STATUS : 010 / OUT COUNT : 1
--------------------------------------------------------------
최원준 / PITCH COUNT :  4
3 .  문보경 : SO / BASE STATUS : 010 / OUT COUNT : 2
--------------------------------------------------------------
최원준 / PITCH COUNT :  7
4 .  오스틴 : 1B / BASE STATUS : 100 / OUT COUNT : 2
--------------------------------------------------------------
최원준 / PITCH COUNT :  8
5 .  박동원 : 1B / BASE STATUS : 110 / OUT COUNT : 2
--------------------------------------------------------------
최원준 / PITCH COUNT :  15
6 .  송찬의 : G5 / BASE STATUS : 110 / OUT COUNT : 3
--------------------------------------------------------------
SCORE :  1
LG :  1 VS DS :  0
 
 
[ Bottom of 1 INNING ]
임찬규 / PITCH COUNT :  0
1 .  정수빈 : 1B / BASE STATUS : 100 / OUT COUNT : 0
-----------------------------

(    1  2  3  4  5  6  7  8  9  R   H  B  E
 LG  1  0  0  1  3  0  1  0  0  6  11  1  0
 DS  1  0  0  0  0  0  0  2  0  3  10  5  2,
 [['OUT', '2B', 'SO', '1B', '1B', 'OUT'],
  ['ERROR', 'OUT', 'SO'],
  ['OUT', 'OUT', 'SO'],
  ['OUT', 'HR', 'OUT', 'OUT'],
  ['OUT', '1B', '1B', '1B', 'OUT', 'BB', '1B', 'OUT'],
  ['OUT', 'OUT', 'SO'],
  ['1B', 'OUT', 'OUT', '1B', 'ERROR', 'OUT'],
  ['SO', 'OUT', '1B', 'OUT'],
  ['OUT', 'OUT', 'OUT']],
 [['1B', 'SO', '1B', 'OUT', 'SO'],
  ['OUT', 'OUT', 'OUT'],
  ['OUT', 'OUT', 'OUT'],
  ['SO', 'BB', 'OUT', '1B', 'BB', 'OUT'],
  ['OUT', 'OUT', 'SO'],
  ['1B', 'SO', 'SO', 'OUT'],
  ['1B', '1B', 'OUT', 'BB', 'OUT', 'SO'],
  ['SO', 'OUT', '1B', '3B', 'BB', '1B', 'OUT'],
  ['BB', '1B', 'SO', 'OUT', 'OUT']],
 ['임찬규', '임찬규', '임찬규', '임찬규', '임찬규', '임찬규', '임찬규', '장현식', '김진성'],
 ['최원준', '최원준', '최원준', '최원준', '최원준', '홍민규', '홍민규', '고효준', '고효준'],
 [[0, 0, 0, 1, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 1, 0, 0],
  [0, 0, 0, 1, 0, 0, 2, 0],
  [0, 0, 0],
  [0, 0, 0, 0, 1, 0

In [46]:
LG_total_wins,DS_total_wins,LG_score_list,DS_score_list, sorted_scores = game_simulation("LG","DS",LG_roster,DS_roster,3,4,"LG",1,500 )     



--------------------------------------------
--------------------------------------------
Game Simulation [ 1 ]
LG wins by 59.0 %
DS wins by 41.0 %
--------------------------------------------
--------------------------------------------



In [38]:
#print(LG_total_wins)
#print(DS_total_wins)
sorted_scores

[('5-3', 6),
 ('2-3', 5),
 ('1-4', 5),
 ('3-5', 5),
 ('5-4', 3),
 ('5-6', 3),
 ('3-4', 3),
 ('2-4', 3),
 ('4-5', 3),
 ('2-5', 2),
 ('7-6', 2),
 ('2-6', 2),
 ('7-8', 2),
 ('8-1', 2),
 ('3-0', 2),
 ('3-9', 2),
 ('4-9', 2),
 ('11-9', 2),
 ('0-4', 2),
 ('6-4', 2),
 ('5-1', 2),
 ('5-2', 2),
 ('7-3', 2),
 ('0-5', 2),
 ('2-1', 2),
 ('6-8', 2),
 ('7-4', 2),
 ('8-2', 1),
 ('9-7', 1),
 ('5-9', 1),
 ('3-2', 1),
 ('5-0', 1),
 ('10-1', 1),
 ('1-5', 1),
 ('2-7', 1),
 ('5-13', 1),
 ('5-7', 1),
 ('9-5', 1),
 ('1-10', 1),
 ('13-7', 1),
 ('2-0', 1),
 ('9-10', 1),
 ('1-0', 1),
 ('3-6', 1),
 ('0-3', 1),
 ('10-16', 1),
 ('1-7', 1),
 ('1-2', 1),
 ('5-8', 1),
 ('4-8', 1),
 ('6-1', 1),
 ('6-2', 1),
 ('3-12', 1),
 ('7-0', 1),
 ('1-6', 1)]

In [39]:
# One Season


team_list = ['LG','SS','LT','DS','SSG','KIA','KW','KT','NC','HH']
team_roster_list =  [LG_roster,
                     SS_roster,
                     LT_roster,
                     DS_roster,
                     SSG_roster,
                     KIA_roster,
                     KW_roster,
                     KT_roster,
                     NC_roster,
                     HH_roster]

win_list = [0,0,0,0,0,0,0,0,0,0]
lose_list = [0,0,0,0,0,0,0,0,0,0]


for i in range(0,10):
    for j in range(i+1,10) :
        pitcher_order_1 = [1,1,1,2,2,2,3,3,3,4,4,4,5,5,5]
        pitcher_order_2 = [1,1,1,2,2,2,3,3,3,4,4,4,5,5,5]
        random.shuffle(pitcher_order_1)
        random.shuffle(pitcher_order_2)
        
        pitcher_order_1.append( np.random.randint(1,6) )
        pitcher_order_2.append( np.random.randint(1,6) )

        
        for idx in range(0,16) :

            
            team1_name = team_list[i]
            team2_name = team_list[j]
            team1_roster = team_roster_list[i]
            team2_roster = team_roster_list[j]
            p_idx_1 = pitcher_order_1[idx]
            p_idx_2 = pitcher_order_2[idx]
            
            
            res = single_game(team1_name,team2_name,team1_roster,team2_roster,p_idx_1,p_idx_2,team1_name,print_record = False,print_HBE = False)
            score_box = res[0]
            
            if score_box.R[team1_name] > score_box.R[team2_name] :
                win_list[i] += 1
                lose_list[j] += 1
            elif score_box.R[team1_name] < score_box.R[team2_name] :
                win_list[j] += 1
                lose_list[i] += 1

In [40]:
def kbo():

    team_list = ['LG','SS','LT','DS','SSG','KIA','KW','KT','NC','HH']
    team_roster_list =  [LG_roster,
                         SS_roster,
                         LT_roster,
                         DS_roster,
                         SSG_roster,
                         KIA_roster,
                         KW_roster,
                         KT_roster,
                         NC_roster,
                         HH_roster]
    
    win_list = [0,0,0,0,0,0,0,0,0,0]
    lose_list = [0,0,0,0,0,0,0,0,0,0]
    
    
    for i in range(0,10):
        for j in range(i+1,10) :
            pitcher_order_1 = [1,1,1,2,2,2,3,3,3,4,4,4,5,5,5]
            pitcher_order_2 = [1,1,1,2,2,2,3,3,3,4,4,4,5,5,5]
            random.shuffle(pitcher_order_1)
            random.shuffle(pitcher_order_2)
            
            pitcher_order_1.append( np.random.randint(1,6) )
            pitcher_order_2.append( np.random.randint(1,6) )
    
            
            for idx in range(0,16) :
    
                
                team1_name = team_list[i]
                team2_name = team_list[j]
                team1_roster = team_roster_list[i]
                team2_roster = team_roster_list[j]
                p_idx_1 = pitcher_order_1[idx]
                p_idx_2 = pitcher_order_2[idx]
                
                
                res = single_game(team1_name,team2_name,team1_roster,team2_roster,p_idx_1,p_idx_2,team1_name,print_record = False,print_HBE = False)
                score_box = res[0]
                
                if score_box.R[team1_name] > score_box.R[team2_name] :
                    win_list[i] += 1
                    lose_list[j] += 1
                elif score_box.R[team1_name] < score_box.R[team2_name] :
                    win_list[j] += 1
                    lose_list[i] += 1
                    
    df = pd.DataFrame([win_list,lose_list ]).T
    df.index = team_list 
    df.columns = ["Win","Lose"]
    
    df["Rank"] = df["Win"].rank(method="min", ascending=False).astype(int)
    rank_list = list ( df["Rank"] )
    df = df.drop("Rank",axis=1)
    
    df = df.sort_values("Win",ascending=False)

    
                    
    return win_list,lose_list,rank_list,df

# 1번 시즌 시뮬레이션

In [52]:
win_list,lose_list,rank_list,df = kbo()
df

,Win,Lose
LG,93,51
HH,85,59
SS,79,65
LT,78,66
NC,73,71
DS,70,74
SSG,68,76
KIA,67,77
KT,57,87
KW,50,94


# 1000 시즌 시뮬레이션

In [42]:
n_repeat = 1
total_win_list = [0,0,0,0,0,0,0,0,0,0]

total_rank_list = [
    [],[],[],[],[],[],[],[],[],[]
]

for k in range(n_repeat):
    win_list,lose_list,rank_list,df = kbo()
    for i in range(10):
        total_win_list[i] += win_list[i]
        total_rank_list[i].append(rank_list[i])
    print(k)
    
df_mean = pd.DataFrame([total_win_list])
df_mean.index = ["Win"]
df_mean.columns = team_list
df_mean = df_mean.T

df_mean.Win = df_mean.Win/ n_repeat
#df_mean = df_mean.sort_values("Win",ascending=False)
df_mean['Lose'] = 144 - df_mean.Win

df_rank = pd.DataFrame(index=team_list, columns=range(1, 11))
df_rank[:] = 0  # 모든 값을 0으로 초기화

# 각 팀별 빈도 채우기
for i, rank_list in enumerate(total_rank_list):
    for rank in rank_list:
        df_rank.iloc[i, rank - 1] += 1  # rank-1은 인덱스 맞추기 위해
df_rank = df_rank / n_repeat * 100
df_rank = df_rank.applymap(lambda x: f"{x:.0f}%" if x > 0 else "0%")

df_total = pd.concat([df_mean,df_rank],axis=1)
df_total = df_total.sort_values("Win",ascending=False)

0


/var/folders/gl/bjnm92ns3yzcwgxyctp7vpm40000gn/T/ipykernel_50802/2121875803.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_rank = df_rank.applymap(lambda x: f"{x:.0f}%" if x > 0 else "0%")


In [43]:
df_total.index = ['LG',
                 '삼성',
                 '롯데',
                 '한화',
                 'KIA',
                 'KT',
                 'NC',
                 '두산',
                 'SSG',
                 '키움']
df_total

,Win,Lose,1,2,3,4,5,6,7,8,9,10
LG,100.0,44.0,100%,0%,0%,0%,0%,0%,0%,0%,0%,0%
삼성,82.0,62.0,0%,100%,0%,0%,0%,0%,0%,0%,0%,0%
롯데,79.0,65.0,0%,0%,100%,0%,0%,0%,0%,0%,0%,0%
한화,74.0,70.0,0%,0%,0%,100%,0%,0%,0%,0%,0%,0%
KIA,74.0,70.0,0%,0%,0%,100%,0%,0%,0%,0%,0%,0%
KT,69.0,75.0,0%,0%,0%,0%,0%,100%,0%,0%,0%,0%
NC,67.0,77.0,0%,0%,0%,0%,0%,0%,100%,0%,0%,0%
두산,66.0,78.0,0%,0%,0%,0%,0%,0%,0%,100%,0%,0%
SSG,58.0,86.0,0%,0%,0%,0%,0%,0%,0%,0%,100%,0%
키움,51.0,93.0,0%,0%,0%,0%,0%,0%,0%,0%,0%,100%
